In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

project_dir = Path(r"C:\Multimodal_KIRC_Project")
rnaseq_dir = project_dir / "01_raw_downloads" / "RNAseq"
processed_dir = project_dir / "03_processed_data"

selected_profiles = pd.read_csv(
    processed_dir / "TCGA_KIRC_selected_patient_level_RNAseq_profiles.csv"
)

print("Selected profiles:", len(selected_profiles))
print("Unique patients:", selected_profiles["Case ID"].nunique())

assert len(selected_profiles) == 533
assert selected_profiles["Case ID"].nunique() == 533

Selected profiles: 533
Unique patients: 533


In [2]:
def find_expression_file(file_name):
    matches = list(rnaseq_dir.rglob(file_name))

    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find: {file_name}")

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple copies found for {file_name}: {matches}"
        )

    return matches[0]

In [4]:
file_paths = []

for _, row in selected_profiles.iterrows():
    path = find_expression_file(row["File Name"])

    file_paths.append({
        "Case ID": row["Case ID"],
        "File ID": row["File ID"],
        "File Name": row["File Name"],
        "Path": str(path)
    })

file_inventory = pd.DataFrame(file_paths)

print("Files located:", len(file_inventory))
print("Unique patients:", file_inventory["Case ID"].nunique())

assert len(file_inventory) == 533
assert file_inventory["Case ID"].nunique() == 533

display(file_inventory.head())

Files located: 533
Unique patients: 533


,Case ID,File ID,File Name,Path
0,TCGA-3Z-A93Z,21702617-18af-41b6-98fa-54395bfd0d9a,c3b9c2b0-bd3d-43ff-a17f-797122d5ac3c.rna_seq.a...,C:\Multimodal_KIRC_Project\01_raw_downloads\RN...
1,TCGA-6D-AA2E,44dade70-3ecc-4be0-88a8-772cb763d5bb,e4497835-a45f-488e-905c-b06d578efef8.rna_seq.a...,C:\Multimodal_KIRC_Project\01_raw_downloads\RN...
2,TCGA-A3-3306,c1215d68-5b15-44c3-9d25-73a77b0b5f31,55599384-574d-44e9-aa61-96d27b54aff1.rna_seq.a...,C:\Multimodal_KIRC_Project\01_raw_downloads\RN...
3,TCGA-A3-3307,b605224a-cf9c-4092-beb4-5ed43c1179de,78855102-69a0-454f-b67a-c0919dc154fb.rna_seq.a...,C:\Multimodal_KIRC_Project\01_raw_downloads\RN...
4,TCGA-A3-3308,7b34b9d4-8526-4738-87b4-f9435a1c7ae1,598575d0-6dc4-4d59-be17-5dff01e6ca52.rna_seq.a...,C:\Multimodal_KIRC_Project\01_raw_downloads\RN...


In [5]:
def read_gene_counts(file_path):
    df = pd.read_csv(
        file_path,
        sep="\t",
        comment="#"
    )

    # Keep real Ensembl gene rows only
    df = df[
        df["gene_id"].astype(str).str.startswith("ENSG")
    ].copy()

    # Keep annotation + raw unstranded counts
    return df[
        [
            "gene_id",
            "gene_name",
            "gene_type",
            "unstranded"
        ]
    ].copy()

In [6]:
test_path = Path(file_inventory.iloc[0]["Path"])

test_counts = read_gene_counts(test_path)

print("Genes in test file:", len(test_counts))
print(test_counts.dtypes)

display(test_counts.head())

Genes in test file: 60660
gene_id         str
gene_name       str
gene_type       str
unstranded    int64
dtype: object


,gene_id,gene_name,gene_type,unstranded
4,ENSG00000000003.15,TSPAN6,protein_coding,1497
5,ENSG00000000005.6,TNMD,protein_coding,20
6,ENSG00000000419.13,DPM1,protein_coding,1318
7,ENSG00000000457.14,SCYL3,protein_coding,247
8,ENSG00000000460.17,C1orf112,protein_coding,91


In [7]:
count_series = []
gene_annotation = None

for i, row in file_inventory.iterrows():

    counts = read_gene_counts(Path(row["Path"]))

    # Save annotation from the first file
    if gene_annotation is None:
        gene_annotation = counts[
            ["gene_id", "gene_name", "gene_type"]
        ].copy()

    # Safety check: gene order must be identical across files
    else:
        if not counts["gene_id"].equals(gene_annotation["gene_id"]):
            raise ValueError(
                f"Gene order mismatch detected for {row['Case ID']}"
            )

    s = counts.set_index("gene_id")["unstranded"]
    s.name = row["Case ID"]

    count_series.append(s)

gene_count_matrix = pd.concat(
    count_series,
    axis=1
).T

gene_count_matrix.index.name = "Case ID"

print("Gene-count matrix shape:", gene_count_matrix.shape)
print("Unique patients:", gene_count_matrix.index.nunique())
print("Missing values:", gene_count_matrix.isna().sum().sum())

display(gene_count_matrix.iloc[:5, :10])

Gene-count matrix shape: (533, 60660)
Unique patients: 533
Missing values: 0


gene_id,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14
Case ID,,,,,,,,,,
TCGA-3Z-A93Z,1497,20,1318,247,91,689,1803,2320,951,670
TCGA-6D-AA2E,2321,1,1104,377,105,410,1503,7177,1365,686
TCGA-A3-3306,2995,3,1444,418,156,858,310,1656,963,772
TCGA-A3-3307,2614,12,1552,440,165,1491,691,2153,2000,1353
TCGA-A3-3308,2865,6,1394,637,195,1046,8585,2781,1270,1543


In [8]:
assert gene_count_matrix.shape[0] == 533
assert gene_count_matrix.index.nunique() == 533
assert gene_count_matrix.isna().sum().sum() == 0
assert (gene_count_matrix >= 0).all().all()

print("All expression-matrix integrity checks passed.")

All expression-matrix integrity checks passed.


In [10]:
gene_count_matrix.to_csv(
    processed_dir / "TCGA_KIRC_raw_gene_count_matrix.csv"
)

gene_annotation.to_csv(
    processed_dir / "TCGA_KIRC_gene_annotation.csv",
    index=False
)

In [11]:
print("Raw matrix shape:", gene_count_matrix.shape)

library_sizes = gene_count_matrix.sum(axis=1)

print("\nLibrary-size summary:")
display(library_sizes.describe())

zero_fraction = (gene_count_matrix == 0).mean(axis=0)

print("\nGene zero-fraction summary:")
display(zero_fraction.describe())

Raw matrix shape: (533, 60660)

Library-size summary:


count    5.330000e+02
mean     5.866454e+07
std      1.444953e+07
min      4.627002e+06
25%      4.867124e+07
50%      5.852197e+07
75%      6.756275e+07
max      1.411188e+08
dtype: float64


Gene zero-fraction summary:


count    60660.000000
mean         0.434042
std          0.403243
min          0.000000
25%          0.001876
50%          0.369606
75%          0.870544
max          1.000000
dtype: float64

In [12]:
min_count = 10
min_fraction = 0.20
min_patients = int(np.ceil(min_fraction * gene_count_matrix.shape[0]))

keep_genes = (
    (gene_count_matrix >= min_count).sum(axis=0)
    >= min_patients
)

filtered_count_matrix = gene_count_matrix.loc[:, keep_genes].copy()

print("Filtering rule:")
print(f"Count >= {min_count} in at least {min_patients} patients")

print("\nGenes before filtering:", gene_count_matrix.shape[1])
print("Genes after filtering:", filtered_count_matrix.shape[1])
print("Genes removed:", gene_count_matrix.shape[1] - filtered_count_matrix.shape[1])

Filtering rule:
Count >= 10 in at least 107 patients

Genes before filtering: 60660
Genes after filtering: 23996
Genes removed: 36664


In [13]:
retained_gene_ids = set(filtered_count_matrix.columns)

retained_annotation = gene_annotation[
    gene_annotation["gene_id"].isin(retained_gene_ids)
].copy()

display(
    retained_annotation["gene_type"]
    .value_counts()
    .rename_axis("gene_type")
    .reset_index(name="N")
    .head(30)
)

,gene_type,N
0,protein_coding,16189
1,lncRNA,5148
2,processed_pseudogene,1109
3,transcribed_unprocessed_pseudogene,402
4,TEC,315
5,unprocessed_pseudogene,194
6,transcribed_processed_pseudogene,150
7,IG_V_gene,112
8,TR_V_gene,73
9,transcribed_unitary_pseudogene,67


In [14]:
protein_coding_ids = set(
    retained_annotation.loc[
        retained_annotation["gene_type"] == "protein_coding",
        "gene_id"
    ]
)

protein_count_matrix = filtered_count_matrix[
    [
        gene_id
        for gene_id in filtered_count_matrix.columns
        if gene_id in protein_coding_ids
    ]
].copy()

print("Filtered genes:", filtered_count_matrix.shape[1])
print("Protein-coding genes retained:", protein_count_matrix.shape[1])

Filtered genes: 23996
Protein-coding genes retained: 16189


In [15]:
library_sizes = protein_count_matrix.sum(axis=1)

cpm_matrix = protein_count_matrix.div(
    library_sizes,
    axis=0
) * 1_000_000

log_cpm_matrix = np.log2(cpm_matrix + 1)

print("log2(CPM + 1) matrix shape:", log_cpm_matrix.shape)

display(log_cpm_matrix.iloc[:5, :10])

log2(CPM + 1) matrix shape: (533, 16189)


gene_id,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14
Case ID,,,,,,,,,,
TCGA-3Z-A93Z,5.598157,0.708244,5.418472,3.142022,1.957489,4.513169,5.861418,6.219603,4.960598,4.474616
TCGA-6D-AA2E,5.919404,0.036532,4.873439,3.415254,1.884697,3.525388,5.305418,7.531819,5.170158,4.216661
TCGA-A3-3306,6.274765,0.106421,5.242156,3.544313,2.316389,4.516936,3.155529,5.434900,4.676607,4.371562
TCGA-A3-3307,5.980485,0.362065,5.243905,3.518497,2.299358,5.187613,4.123209,5.705458,5.601223,5.051527
TCGA-A3-3308,5.975793,0.175949,4.960477,3.884652,2.382799,4.561464,7.543721,5.933553,4.830592,5.102503


In [16]:
assert log_cpm_matrix.shape[0] == 533
assert log_cpm_matrix.isna().sum().sum() == 0
assert np.isfinite(log_cpm_matrix.to_numpy()).all()

print("All normalized-expression integrity checks passed.")

All normalized-expression integrity checks passed.


In [17]:
# Save reference matrices and filtering audit

log_cpm_matrix.to_csv(
    processed_dir / "TCGA_KIRC_reference_log2CPM_protein_coding_matrix.csv"
)

protein_count_matrix.to_csv(
    processed_dir / "TCGA_KIRC_reference_protein_coding_count_matrix.csv"
)

filtering_summary = pd.DataFrame({
    "Step": [
        "Raw Ensembl genes",
        "Expression-prevalence filtered",
        "Protein-coding retained"
    ],
    "Number of genes": [
        gene_count_matrix.shape[1],
        filtered_count_matrix.shape[1],
        protein_count_matrix.shape[1]
    ]
})

display(filtering_summary)

filtering_summary.to_csv(
    processed_dir / "TCGA_KIRC_gene_filtering_summary.csv",
    index=False
)

,Step,Number of genes
0,Raw Ensembl genes,60660
1,Expression-prevalence filtered,23996
2,Protein-coding retained,16189


In [18]:
protein_annotation = gene_annotation[
    gene_annotation["gene_type"] == "protein_coding"
].copy()

protein_annotation.to_csv(
    processed_dir / "TCGA_KIRC_protein_coding_gene_annotation.csv",
    index=False
)

print("Protein-coding annotation rows:", len(protein_annotation))

Protein-coding annotation rows: 19962
